In [ ]:

# from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import sys
sys.path.append(r"D:\HopeAI\Assignments\MyModules")

from data_analysis_utils import Preprocessing

stocks_df = pd.read_csv(r"D:\HopeAI\Assignments\ML and DS Capstone\1. Data Collection\nifty50_stocks.csv")
nifty_df = pd.read_csv(r'D:\HopeAI\Assignments\ML and DS Capstone\1. Data Collection\NIFTY50.csv')
stocks_df


,Date,Close,High,Low,Open,Volume,Stock,Sector,Industry,Market Cap
0,2014-01-01,145.392685,146.409103,144.922719,146.015651,4492436,HDFCBANK,Financial Services,Banks - Regional,14818861907968
1,2014-01-02,143.611237,147.491144,142.857111,145.359927,7228992,HDFCBANK,Financial Services,Banks - Regional,14818861907968
2,2014-01-03,144.944565,145.578470,142.081090,142.518262,6513184,HDFCBANK,Financial Services,Banks - Regional,14818861907968
3,2014-01-06,144.638565,144.922725,143.611207,144.922725,5963420,HDFCBANK,Financial Services,Banks - Regional,14818861907968
4,2014-01-07,145.228760,146.660498,142.802444,145.818932,7896920,HDFCBANK,Financial Services,Banks - Regional,14818861907968
...,...,...,...,...,...,...,...,...,...,...
126939,2024-12-23,571.339233,585.429953,569.337575,575.917235,4691250,SHRIRAMFIN,Financial Services,Credit Services,1214284562432
126940,2024-12-24,571.349121,574.252482,565.829742,571.358979,2020780,SHRIRAMFIN,Financial Services,Credit Services,1214284562432
126941,2024-12-26,580.059204,581.852750,569.010528,573.202120,4462340,SHRIRAMFIN,Financial Services,Credit Services,1214284562432
126942,2024-12-27,574.510193,582.625693,572.845410,582.041089,1524915,SHRIRAMFIN,Financial Services,Credit Services,1214284562432


In [9]:

stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
stocks_df['Year'] = stocks_df['Date'].dt.year

nifty_df['Date'] = pd.to_datetime(nifty_df['Date'])
nifty_df.set_index('Date', inplace=True)
nifty_returns = nifty_df['Close'].pct_change().dropna()
nifty_returns

feature_rows = []
stocks = stocks_df['Stock'].unique()
stocks

for stock in stocks:
    stock_data = stocks_df[stocks_df['Stock'] == stock].copy()
    
    for year, year_data in stock_data.groupby('Year'):
        daily_returns = year_data['Close'].pct_change().dropna()
        monthly_prices = year_data.set_index('Date')['Close'].resample('ME').ffill()
        monthly_returns = monthly_prices.pct_change().dropna()
        

        # Collected features
        sector = year_data['Sector'].unique().item()
        industry = year_data['Industry'].unique().item()
        market_cap = year_data['Market Cap'].unique().item()

        # Return features
        total_annual_return = (year_data['Close'].iloc[-1] / year_data['Close'].iloc[0]) - 1
        avg_monthly_return = monthly_returns.mean()
        std_monthly_return = monthly_returns.std()
        
        
        # Volatility / Price Range
        max_drawdown = ((year_data['Close'].cummax() - year_data['Close']) / year_data['Close'].cummax()).max()
        std_daily_return = daily_returns.std()
        pos_days_pct = (daily_returns > 0).mean()
        neg_days_pct = (daily_returns < 0).mean()
        avg_daily_TR = (year_data['High'] - year_data['Low']).mean()
        max_daily_TR = (year_data['High'] - year_data['Low']).max()
        
        # Volume
        avg_volume = year_data['Volume'].mean()
        volume_spike_pct = (year_data['Volume'] > 2 * avg_volume).mean()
        
        # Trend / Momentum
        twelve_month_momentum = (monthly_prices.iloc[-1] - monthly_prices.iloc[0]) / monthly_prices.iloc[0]
        months_pos_pct = (monthly_returns > 0).mean()
        
        # Market relationship
        year_data = year_data.set_index('Date').sort_index()
        daily_returns = year_data['Close'].pct_change().dropna()
        nifty_year = nifty_returns.loc[daily_returns.index.min(): daily_returns.index.max()]

        combined = pd.DataFrame({
            'Stock_Return': daily_returns,
            'NIFTY_Return': nifty_year
        }).dropna()

        if len(combined) > 30:  # require at least 30 days
            cov_matrix = np.cov(combined['Stock_Return'], combined['NIFTY_Return'])
            beta = cov_matrix[0,1] / cov_matrix[1,1] if cov_matrix[1,1] != 0 else np.nan
            corr_nifty = combined['Stock_Return'].corr(combined['NIFTY_Return'])
        else:
            beta = np.nan
            corr_nifty = np.nan

        feature_row = {
            'Stock': stock,
            'Sector': sector,
            'Industry': industry,
            'Market Cap': market_cap,
            'Year': year,
            'Total_Annual_Return': total_annual_return*100,  # Percentage
            'Avg_Monthly_Return': avg_monthly_return*100,  # Percentage
            'Std_Monthly_Return': std_monthly_return*100,  # Percentage
            'Max_Drawdown': max_drawdown*100,  # Percentage
            'Std_Daily_Return': std_daily_return*100,  # Percentage
            'Pos_Days_Pct': pos_days_pct*100,  # Percentage
            'Neg_Days_Pct': neg_days_pct*100,  # Percentage
            'Avg_Daily_TR': avg_daily_TR,
            'Max_Daily_TR': max_daily_TR,
            'Avg_Volume': avg_volume,
            'Volume_Spike_Pct': volume_spike_pct*100,  # Percentage
            '12M_Momentum': twelve_month_momentum*100,  # Percentage
            'Months_Pos_Pct': months_pos_pct*100,  # Percentage
            'Beta_vs_NIFTY': beta,
            'Corr_with_NIFTY': corr_nifty
        }
        
        feature_rows.append(feature_row)

# Create DataFrame
features_df = pd.DataFrame(feature_rows)

# Save feature matrix
features_df.to_csv('stock_risk_features.csv', index=False)
features_df



,Stock,Sector,Industry,Market Cap,Year,Total_Annual_Return,Avg_Monthly_Return,Std_Monthly_Return,Max_Drawdown,Std_Daily_Return,Pos_Days_Pct,Neg_Days_Pct,Avg_Daily_TR,Max_Daily_TR,Avg_Volume,Volume_Spike_Pct,12M_Momentum,Months_Pos_Pct,Beta_vs_NIFTY,Corr_with_NIFTY
0,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2014,45.506307,4.090170,4.473521,7.942126,1.244001,51.440329,47.736626,3.540610,20.284804,8.008632e+06,6.557377,53.991277,81.818182,1.019285,0.655653
1,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2015,14.519620,0.189036,4.196735,13.123291,1.171941,49.387755,50.612245,4.371720,10.960170,6.271229e+06,2.845528,1.200524,54.545455,0.886184,0.778026
2,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2016,11.696905,1.463277,5.118095,13.419040,0.989180,50.204082,49.795918,4.093717,22.421054,5.565438e+06,4.878049,15.835598,63.636364,0.754961,0.729038
3,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2017,57.450656,3.575875,3.175871,4.512343,0.843602,60.323887,39.271255,4.948708,21.125941,7.546210e+06,5.241935,46.492195,90.909091,0.728813,0.492274
4,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2018,15.879063,0.792284,6.092820,13.187105,1.061314,50.204082,49.795918,7.021670,27.411052,9.757692e+06,6.097561,7.143505,45.454545,0.686503,0.524979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,SHRIRAMFIN,Financial Services,Credit Services,1214284562432,2020,-5.316279,3.844171,26.413013,66.297624,4.904156,50.800000,49.200000,8.414907,28.721044,2.306230e+07,8.366534,5.993246,63.636364,1.591980,0.640662
515,SHRIRAMFIN,Financial Services,Credit Services,1214284562432,2021,15.140208,-0.169706,7.399586,29.229798,2.822868,48.987854,51.012146,9.704703,35.646977,1.043849e+07,7.661290,-4.520160,36.363636,1.305706,0.457003
516,SHRIRAMFIN,Financial Services,Credit Services,1214284562432,2022,11.323448,1.366981,6.926183,22.148332,2.348058,52.226721,47.773279,7.991183,23.780831,5.312576e+06,4.838710,13.348920,63.636364,1.123323,0.521125
517,SHRIRAMFIN,Financial Services,Credit Services,1214284562432,2023,53.461458,4.851050,8.271529,12.437997,1.853198,53.278689,46.311475,8.139383,34.489616,5.692273e+06,7.346939,63.504551,72.727273,0.961684,0.321195


In [12]:
features_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 519 entries, 0 to 518
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Stock                519 non-null    object 
 1   Sector               519 non-null    object 
 2   Industry             519 non-null    object 
 3   Market Cap           519 non-null    int64  
 4   Year                 519 non-null    int64  
 5   Total_Annual_Return  519 non-null    float64
 6   Avg_Monthly_Return   519 non-null    float64
 7   Std_Monthly_Return   517 non-null    float64
 8   Max_Drawdown         519 non-null    float64
 9   Std_Daily_Return     519 non-null    float64
 10  Pos_Days_Pct         519 non-null    float64
 11  Neg_Days_Pct         519 non-null    float64
 12  Avg_Daily_TR         519 non-null    float64
 13  Max_Daily_TR         519 non-null    float64
 14  Avg_Volume           519 non-null    float64
 15  Volume_Spike_Pct     519 non-null    flo

In [14]:
print(features_df.isnull().mean()*100)


Stock                  0.000000
Sector                 0.000000
Industry               0.000000
Market Cap             0.000000
Year                   0.000000
Total_Annual_Return    0.000000
Avg_Monthly_Return     0.000000
Std_Monthly_Return     0.385356
Max_Drawdown           0.000000
Std_Daily_Return       0.000000
Pos_Days_Pct           0.000000
Neg_Days_Pct           0.000000
Avg_Daily_TR           0.000000
Max_Daily_TR           0.000000
Avg_Volume             0.000000
Volume_Spike_Pct       0.000000
12M_Momentum           0.000000
Months_Pos_Pct         0.000000
Beta_vs_NIFTY          0.192678
Corr_with_NIFTY        0.192678
dtype: float64


In [17]:
quan, qual = Preprocessing.quanQual(features_df)

print(quan, qual)


Index(['Market Cap', 'Year', 'Total_Annual_Return', 'Avg_Monthly_Return',
       'Std_Monthly_Return', 'Max_Drawdown', 'Std_Daily_Return',
       'Pos_Days_Pct', 'Neg_Days_Pct', 'Avg_Daily_TR', 'Max_Daily_TR',
       'Avg_Volume', 'Volume_Spike_Pct', '12M_Momentum', 'Months_Pos_Pct',
       'Beta_vs_NIFTY', 'Corr_with_NIFTY'],
      dtype='object') Index(['Stock', 'Sector', 'Industry'], dtype='object')


### Missing Values


In [ ]:
dataset = Preprocessing.simple_missing(features_df, quan, qual) 
dataset


,Stock,Sector,Industry,Market Cap,Year,Total_Annual_Return,Avg_Monthly_Return,Std_Monthly_Return,Max_Drawdown,Std_Daily_Return,Pos_Days_Pct,Neg_Days_Pct,Avg_Daily_TR,Max_Daily_TR,Avg_Volume,Volume_Spike_Pct,12M_Momentum,Months_Pos_Pct,Beta_vs_NIFTY,Corr_with_NIFTY
0,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2014,45.506307,4.090170,4.473521,7.942126,1.244001,51.440329,47.736626,3.540610,20.284804,8.008632e+06,6.557377,53.991277,81.818182,1.019285,0.655653
1,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2015,14.519620,0.189036,4.196735,13.123291,1.171941,49.387755,50.612245,4.371720,10.960170,6.271229e+06,2.845528,1.200524,54.545455,0.886184,0.778026
2,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2016,11.696905,1.463277,5.118095,13.419040,0.989180,50.204082,49.795918,4.093717,22.421054,5.565438e+06,4.878049,15.835598,63.636364,0.754961,0.729038
3,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2017,57.450656,3.575875,3.175871,4.512343,0.843602,60.323887,39.271255,4.948708,21.125941,7.546210e+06,5.241935,46.492195,90.909091,0.728813,0.492274
4,HDFCBANK,Financial Services,Banks - Regional,14818861907968,2018,15.879063,0.792284,6.092820,13.187105,1.061314,50.204082,49.795918,7.021670,27.411052,9.757692e+06,6.097561,7.143505,45.454545,0.686503,0.524979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,SHRIRAMFIN,Financial Services,Credit Services,1214284562432,2020,-5.316279,3.844171,26.413013,66.297624,4.904156,50.800000,49.200000,8.414907,28.721044,2.306230e+07,8.366534,5.993246,63.636364,1.591980,0.640662
515,SHRIRAMFIN,Financial Services,Credit Services,1214284562432,2021,15.140208,-0.169706,7.399586,29.229798,2.822868,48.987854,51.012146,9.704703,35.646977,1.043849e+07,7.661290,-4.520160,36.363636,1.305706,0.457003
516,SHRIRAMFIN,Financial Services,Credit Services,1214284562432,2022,11.323448,1.366981,6.926183,22.148332,2.348058,52.226721,47.773279,7.991183,23.780831,5.312576e+06,4.838710,13.348920,63.636364,1.123323,0.521125
517,SHRIRAMFIN,Financial Services,Credit Services,1214284562432,2023,53.461458,4.851050,8.271529,12.437997,1.853198,53.278689,46.311475,8.139383,34.489616,5.692273e+06,7.346939,63.504551,72.727273,0.961684,0.321195


In [ ]:
# # Standardize numeric features

# numeric_cols = features_df.select_dtypes(include=np.number).columns.drop(['Year'])
# scaler = StandardScaler()
# features_df[numeric_cols] = scaler.fit_transform(features_df[numeric_cols])
